# Random Forest — Physics vs Context vs All Features
## 5-seed diagnostic for the 49-observation HEA dataset

**Purpose:** determine why Random Forest performs strongly by separating the contribution of:
1. physics/material descriptors,
2. experimental context (temperature and strain rate),
3. all ten features.

This uses the same observation-level 80/20 split protocol as the previous RF seed experiment.

### Feature groups

**Physics/material (8):**
`xAl, VEC, dHmix, dSmix, delta, Omega, dchi, Mean_Tm`

**Context (2):**
`Temperature_C, log10_Strain_rate`

**All (10):**
all features above.

### Controlled settings

- fixed train/test split: `test_size=0.20`, `random_state=42`
- RF: `n_estimators=300`
- RF random seeds: 42–46
- no feature scaling (Random Forest does not require it)
- identical train/test observations for all three feature sets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEEDS = [42, 43, 44, 45, 46]
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
N_ESTIMATORS = 300
TARGET = "UTS_MPa"

PHYSICS_FEATURES = [
    "xAl", "VEC", "dHmix", "dSmix",
    "delta", "Omega", "dchi", "Mean_Tm"
]

CONTEXT_FEATURES = [
    "Temperature_C", "log10_Strain_rate"
]

ALL_FEATURES = PHYSICS_FEATURES + CONTEXT_FEATURES

COMPOSITION_COLS = [
    "Al_norm", "Co_norm", "Cr_norm", "Fe_norm", "Ni_norm"
]

print("Configuration loaded.")

## 1. Load the same Excel workbook (`B_Model_Ready`)

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

df = pd.read_excel(
    file_path,
    sheet_name="B_Model_Ready"
)

required = ALL_FEATURES + [TARGET] + COMPOSITION_COLS
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[required] = df[required].apply(
    pd.to_numeric,
    errors="coerce"
)

df = df.dropna(
    subset=required
).copy()

df["Composition_Group"] = (
    df[COMPOSITION_COLS]
      .round(3)
      .astype(str)
      .agg("_".join, axis=1)
)

print("Dataset shape:", df.shape)
print("Observations:", len(df))
print("Unique compositions:", df["Composition_Group"].nunique())

## 2. Freeze the observation-level train/test split

The same indices are reused for **Physics-only, Context-only, and All-features RF** so the comparison is controlled.

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE
)

print("Training observations:", len(train_idx))
print("Testing observations :", len(test_idx))

train_groups = set(df.iloc[train_idx]["Composition_Group"])
test_groups = set(df.iloc[test_idx]["Composition_Group"])

overlap = train_groups.intersection(test_groups)

print("Training compositions:", len(train_groups))
print("Testing compositions :", len(test_groups))
print("Composition overlap   :", len(overlap))

## 3. Helper function

In [ ]:
def evaluate_rf(feature_list, seed):
    X_train = df.iloc[train_idx][feature_list]
    X_test = df.iloc[test_idx][feature_list]

    y_train = df.iloc[train_idx][TARGET]
    y_test = df.iloc[test_idx][TARGET]

    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=seed,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)

    return {
        "Train_MAE": mean_absolute_error(y_train, train_pred),
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Train_R2": r2_score(y_train, train_pred),
        "Test_MAE": mean_absolute_error(y_test, test_pred),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Test_R2": r2_score(y_test, test_pred),
        "MAE_Gap": mean_absolute_error(y_test, test_pred) - mean_absolute_error(y_train, train_pred),
        "RMSE_Gap": np.sqrt(mean_squared_error(y_test, test_pred)) - np.sqrt(mean_squared_error(y_train, train_pred)),
        "R2_Gap": r2_score(y_train, train_pred) - r2_score(y_test, test_pred),
    }

## 4. Run the 5-seed comparison

In [ ]:
feature_sets = {
    "RF_Physics_Only": PHYSICS_FEATURES,
    "RF_Context_Only": CONTEXT_FEATURES,
    "RF_All_Features": ALL_FEATURES
}

records = []

for model_name, features in feature_sets.items():
    print(f"\n===== {model_name} =====")

    for seed in SEEDS:
        result = evaluate_rf(features, seed)

        records.append({
            "Model": model_name,
            "Seed": seed,
            **result
        })

        print(
            f"Seed {seed}: "
            f"Test MAE={result['Test_MAE']:.3f}, "
            f"Test RMSE={result['Test_RMSE']:.3f}, "
            f"Test R2={result['Test_R2']:.3f}"
        )

rf_context_results = pd.DataFrame(records)

print("\n===== Full 5-seed results =====")
display(rf_context_results)

## 5. Test performance — mean ± SD

In [ ]:
summary = (
    rf_context_results
    .groupby("Model")
    [["Test_MAE", "Test_RMSE", "Test_R2"]]
    .agg(["mean", "std"])
)

display(summary)

print("\nPresentation-ready:")
for model_name in feature_sets:
    row = summary.loc[model_name]
    print(
        f"{model_name}: "
        f"MAE {row[('Test_MAE','mean')]:.2f} ± {row[('Test_MAE','std')]:.2f}; "
        f"RMSE {row[('Test_RMSE','mean')]:.2f} ± {row[('Test_RMSE','std')]:.2f}; "
        f"R2 {row[('Test_R2','mean')]:.3f} ± {row[('Test_R2','std')]:.3f}"
    )

## 6. Train vs test gap

In [ ]:
gap_summary = (
    rf_context_results
    .groupby("Model")
    [["Train_MAE", "Train_RMSE", "Train_R2",
      "Test_MAE", "Test_RMSE", "Test_R2",
      "MAE_Gap", "RMSE_Gap", "R2_Gap"]]
    .agg(["mean", "std"])
)

display(gap_summary)

## 7. Compare test R² across feature groups

In [ ]:
plot_df = (
    rf_context_results
    .groupby("Model")["Test_R2"]
    .agg(["mean", "std"])
    .reindex(feature_sets.keys())
)

plt.figure(figsize=(8, 5))
plt.bar(
    plot_df.index,
    plot_df["mean"],
    yerr=plot_df["std"],
    capsize=5
)
plt.ylabel("Test R²")
plt.xlabel("Feature set")
plt.title("Random Forest: Physics vs Context vs All Features")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Feature importance for the all-feature RF

In [ ]:
importance_records = []

for seed in SEEDS:
    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=seed,
        n_jobs=-1
    )

    rf.fit(
        df.iloc[train_idx][ALL_FEATURES],
        df.iloc[train_idx][TARGET]
    )

    importance_records.append(rf.feature_importances_)

importance_df = pd.DataFrame(
    importance_records,
    columns=ALL_FEATURES
)

importance_summary = pd.DataFrame({
    "Feature": ALL_FEATURES,
    "Mean_Importance": importance_df.mean(axis=0).values,
    "SD_Importance": importance_df.std(axis=0, ddof=1).values
}).sort_values(
    "Mean_Importance",
    ascending=False
)

display(importance_summary)

## 9. Direct diagnostic

Use the following logic when interpreting the result:

- **Context-only close to All-features:** experimental context carries much of the predictive signal.
- **Physics-only close to All-features:** material descriptors carry substantial predictive signal.
- **All-features much better than either subset:** RF is benefiting from complementary physics + context information.
- **All-features much worse than a subset:** the added feature group may be noisy or redundant for this dataset.

Do not infer causality from Random Forest feature importance alone.

In [ ]:
# Save all results
rf_context_results.to_csv(
    "RF_Physics_Context_All_5Seed_Results.csv",
    index=False
)

summary.to_csv(
    "RF_Physics_Context_All_5Seed_Summary.csv"
)

gap_summary.to_csv(
    "RF_Physics_Context_All_5Seed_Gaps.csv"
)

importance_summary.to_csv(
    "RF_All_Feature_Importance_5Seed.csv",
    index=False
)

print("Saved CSV result files.")